# Text mining sur les narratifs techniques

## Objectif de ce notebook

Dernier bloc d'analyse : au lieu de compter des codes, on regarde directement le **texte libre** écrit par les techniciens (colonne `Discrepancy`) pour faire émerger les mots et expressions les plus caractéristiques de chaque chapitre ATA. L'idée : deux chapitres peuvent avoir le même nombre de signalements, mais des causes complètement différentes — ce texte permet de voir ça.

**Nouveauté technique** : on utilise `scikit-learn`, une bibliothèque de machine learning, pour une méthode appelée **TF-IDF** (expliquée plus bas).

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from sdr_analytics import config
from sdr_analytics.metrics import pareto_table
from sdr_analytics.text_clean import MOTS_VIDES_ANGLAIS, nettoyer_texte

df = pd.read_parquet(config.CHEMIN_DONNEES_TRAITEES / "sdr_airbus_clean.parquet")
df_texte = df[df["dans_perimetre"] & df["annee_complete"]].copy()
print(f"{len(df_texte)} lignes dans le périmètre")

112195 lignes dans le périmètre


## 1. À quoi ressemble ce texte ?

Rappel du notebook 01 : c'est écrit par des techniciens, en anglais, avec beaucoup d'abréviations. On a construit un petit glossaire (`data/reference/glossaire_abreviations.csv`) pour les principales.

In [2]:
glossaire = pd.read_csv(config.CHEMIN_DONNEES_REFERENCE / "glossaire_abreviations.csv")
glossaire.head(8)

,abreviation,signification
0,IAW,In Accordance With (conformément à)
1,AMM,Aircraft Maintenance Manual (manuel de mainten...
2,R&R,Removed and Replaced (déposé et remplacé)
3,C/W,Complied With (effectué / réalisé conformément...
4,C/A,Corrective Action (action corrective)
5,NDT,Non-Destructive Testing (contrôle non destructif)
6,INSP,Inspection
7,REPL,Replaced (remplacé)


In [3]:
for texte in df_texte["Discrepancy"].sample(3, random_state=1):
    print(texte)
    print("-" * 80)

S1 CHECK, GALVANIC CORROSION ON FLOOR SUPPORT BETWEEN FRAME 75 - 76 AT STRINGER 26L.  REPLACED FLOOR SUPPORT IAW SRM 53-41-14, SRM 51-42-11 & SRM 51-44-11.
--------------------------------------------------------------------------------
FOUND R2 DOOR EMER. EXIT SIGN COVER MISSING, REPLACED MISSING LENS COVER.
--------------------------------------------------------------------------------
POOR SLIDE AT DOOR 841 MUST BE REPLACED DUE TROUBLESHOOTING OF EPSU 10WL EXTERIOR LIGHT FAULT.  REPLACED DOOR SLIDE AT DOOR 841 IAW AMM 25-62-44-400 803-A  AND TEST IS OK.
--------------------------------------------------------------------------------


## 2. Nettoyer le texte avant l'analyse

Deux choses à retirer avant de chercher des mots-clés utiles :
- Les **références/numéros** (numéros de pièce, de série, de log...) : ce ne sont pas des mots, ils n'apportent rien à une analyse thématique.
- Les **mots vides** ("the", "and", "was"...) : très fréquents, mais sans signification propre.

**Attention au piège** : le texte est en anglais, donc il faut une liste de mots vides anglaise — utiliser une liste française par erreur ne retirerait rien du tout, silencieusement.

In [4]:
exemple = df_texte["Discrepancy"].iloc[0]
print("Avant :", exemple)
print()
print("Après :", nettoyer_texte(exemple))

Avant : DURING CLIMB, ODOR OF BURNING RUBBER APPARENT.  SOON AFTER SMOKE EMINATED FROM STANDBY COMPASS HOUSING.  SWITCHED OFF STBY COMPASS LIGHT WHICH WAS SELECTED ON.  CB E75 TRIPPED SHORTLY THEREAFTER.  SMOKE DISSIPATED AFTER STBY COMPASS LIGHT WAS SWITCHED OFF.  DECLARED EMERGENCY & RETURNED TO DEPARTURE.  DUMPED FUEL & LANDED UNDER MGLW AT 3999,500.  ATB REPLACED STANDBY COMPASS IAW AMM 34-22-23.  REPLACED  STANDBY COMPASS INTEGRAL LIGHTING CONNECTOR IAW WDM 33-12-03.  LIGHTING OPS CK OK 23-13-00.

Après : during climb, odor of burning rubber apparent.  soon after smoke eminated from standby compass housing.  switched off stby compass light which was selected on.  cb   tripped shortly thereafter.  smoke dissipated after stby compass light was switched off.  declared emergency & returned to departure.  dumped fuel & landed under mglw at  , .  atb replaced standby compass iaw amm  - - .  replaced  standby compass integral lighting connector iaw wdm  - - .  lighting ops ck ok  - - .


## 3. TF-IDF : trouver les mots caractéristiques, pas juste les plus fréquents

Si on comptait juste les mots les plus fréquents, on retrouverait des mots génériques de maintenance ("found", "removed", "replaced"...) qui apparaissent partout, quel que soit le chapitre — pas très utile.

**TF-IDF** (Term Frequency – Inverse Document Frequency) corrige ça : un mot obtient un score élevé s'il est fréquent *dans un signalement donné*, mais rare *dans l'ensemble des signalements*. Un mot très générique (présent partout) est automatiquement pénalisé, même s'il apparaît souvent.

On calcule ça au niveau de chaque signalement individuel (pas par chapitre directement), et on regarde aussi bien les mots seuls ("crack") que les paires de mots ("fuel leak") — certaines expressions n'ont de sens qu'ensemble.

In [5]:
vectoriseur = TfidfVectorizer(
    preprocessor=nettoyer_texte,
    stop_words=list(MOTS_VIDES_ANGLAIS),
    ngram_range=(1, 2),   # mots seuls + paires de mots
    min_df=10,             # ignorer les mots trop rares (moins de 10 signalements)
    max_df=0.5,            # ignorer les mots trop génériques (présents dans plus de 50% des signalements)
    max_features=3000,
)

matrice_tfidf = vectoriseur.fit_transform(df_texte["Discrepancy"])
termes = vectoriseur.get_feature_names_out()

print(f"Matrice : {matrice_tfidf.shape[0]} signalements x {matrice_tfidf.shape[1]} mots/expressions retenus")

Matrice : 112195 signalements x 3000 mots/expressions retenus


## 4. Les termes les plus caractéristiques par chapitre

Pour chaque chapitre, on prend tous ses signalements et on calcule le score TF-IDF *moyen* de chaque mot sur ce sous-ensemble — les mots avec le score moyen le plus élevé sont ceux qui caractérisent le mieux ce chapitre en particulier.

In [6]:
top8_chapitres = pareto_table(df_texte.groupby("chapitre_ata").size()).head(8).index.tolist()
libelles = df_texte[["chapitre_ata", "libelle_ata"]].drop_duplicates().set_index("chapitre_ata")["libelle_ata"]

lignes_export = []
for chapitre in top8_chapitres:
    masque = (df_texte["chapitre_ata"] == chapitre).values
    scores_moyens = matrice_tfidf[masque].mean(axis=0).A1
    top_indices = scores_moyens.argsort()[::-1][:8]

    print(f"\n{libelles[chapitre]} :")
    for idx in top_indices:
        print(f"  - {termes[idx]} ({scores_moyens[idx]:.3f})")
        lignes_export.append({
            "chapitre_ata": chapitre,
            "libelle_ata": libelles[chapitre],
            "terme": termes[idx],
            "score": float(scores_moyens[idx]),
        })


Fuselage :
  - srm (0.070)
  - floor (0.065)
  - fr (0.056)
  - corrosion (0.048)
  - frame (0.045)
  - iaw srm (0.042)
  - floor panel (0.039)
  - aft (0.039)

Éclairage :
  - emergency (0.111)
  - light (0.106)
  - epsu (0.068)
  - battery (0.067)
  - emergency light (0.064)
  - exit (0.054)
  - amm (0.046)
  - seat (0.046)

Portes :
  - door (0.165)
  - damper (0.065)
  - serviced (0.059)
  - door damper (0.056)
  - pressure (0.054)
  - assist (0.053)
  - cylinder (0.053)
  - bottle (0.051)

Équipements et aménagements :
  - door (0.098)
  - slide (0.090)
  - flashlight (0.057)
  - assist (0.046)
  - amm (0.045)
  - megaphone (0.045)
  - bottle (0.044)
  - door assist (0.043)

Climatisation :
  - odor (0.111)
  - smell (0.085)
  - pack (0.082)
  - cabin (0.056)
  - tsm (0.048)
  - performed (0.044)
  - apu (0.042)
  - dirty (0.039)

Empennages :
  - hinge (0.218)
  - elevator (0.155)
  - hinge arm (0.129)
  - fitting (0.119)
  - arm (0.117)
  - stabilizer (0.116)
  - horizontal (0.

## 5. Vérifier avec les vrais narratifs (drill-down)

Un score TF-IDF, ça reste abstrait. Pour vérifier que ça correspond bien à quelque chose de réel, on prend le terme le plus caractéristique du premier chapitre et on regarde des vrais signalements qui le contiennent.

In [7]:
chapitre_exemple = top8_chapitres[0]
terme_exemple = [l["terme"] for l in lignes_export if l["chapitre_ata"] == chapitre_exemple][0]

print(f"Chapitre : {libelles[chapitre_exemple]} — terme : \"{terme_exemple}\"\n")

correspondances = df_texte[
    (df_texte["chapitre_ata"] == chapitre_exemple)
    & df_texte["Discrepancy"].str.contains(terme_exemple, case=False, na=False)
]

for texte in correspondances["Discrepancy"].head(3):
    print(texte)
    print("-" * 80)

Chapitre : Fuselage — terme : "srm"



DURING INSPECTION, FOUND CORROSION ON WEB IN AFT CARGO BAY AT STRINGER 38R BETWEEN FRAMES 56 - 57.  REPLACED THE WEB IAW SRM 51-72-11 AND DWG D53470320.
--------------------------------------------------------------------------------
DURING INSPECTION, FOUND CORROSION ON STRINGER 26R BETWEEN FRAMES 67 AND 68. REPLACED TEH STRINGER  IAW SRM 51-72-11 AND DWG D53480505.
--------------------------------------------------------------------------------
DURING INSPECTION, FOUND CORROSION ON THE FLOOR SUPPORT -Y81 BETWEEN FRAMES 12 - 16.  REPLACED THE FLOOR SUPPORT IAW SRM 51-72-11, DWG N.D53112828, AND D53110984.
--------------------------------------------------------------------------------


## 6. Exporter pour le dashboard

In [8]:
export_termes = pd.DataFrame(lignes_export)
export_termes.to_parquet(config.CHEMIN_DONNEES_TRAITEES / "agg_top_termes_chapitre.parquet", index=False)
print(f"Exporté : {len(export_termes)} lignes -> data/processed/agg_top_termes_chapitre.parquet")

Exporté : 64 lignes -> data/processed/agg_top_termes_chapitre.parquet


## Ce qu'on retient

- **Le texte confirme et explique le résultat du notebook 05** : "corrosion" ressort comme terme caractéristique à la fois pour le **Fuselage** et la **Voilure** — exactement les deux chapitres identifiés comme "usure/fatigue" (plus signalés sur avions âgés) dans le bloc âge/usure. Le texte donne la raison concrète derrière le chiffre : ce sont majoritairement des inspections qui trouvent de la corrosion, un phénomène qui s'accumule logiquement avec le temps.
- **Chaque chapitre a un vocabulaire très spécifique**, pas juste des synonymes de "panne" : "hinge/elevator/stabilizer" pour les Empennages, "gear/brake/wheel" pour le Train d'atterrissage, "door/damper/cylinder" pour les Portes — le TF-IDF isole bien le sujet propre à chaque chapitre plutôt que le vocabulaire générique de maintenance.
- **Résultat le plus surprenant** : le chapitre Climatisation est dominé par "odor/smell" (odeur) plutôt que par des termes mécaniques classiques — suggère que le problème le plus fréquemment signalé sur ce chapitre n'est pas une panne du système en soi, mais une odeur perçue en cabine (qui peut avoir des causes variées, pas uniquement la climatisation).
- **Vérification par échantillon** : pour le terme le plus caractéristique du Fuselage ("srm" = Structural Repair Manual), les narratifs réels correspondent bien à des réparations de corrosion documentées selon ce manuel — le score TF-IDF ne sort pas de nulle part, il pointe vers un vrai contenu cohérent.

*(Ces points seront repris dans `docs/methodologie_limites.md`.)*